In [ ]:
# Setup: clone or pull logseer repo and import core library
import os, sys
if not os.path.exists("logseer"):
    !git clone https://github.com/masahiko-shibata/logseer.git
else:
    !cd logseer && git pull
sys.path.insert(0, "logseer")
from logseer import Tester
import keras

In [ ]:
# Load model and tokenizer from Google Drive
from google.colab import drive
import tensorflow as tf
import pickle, shutil, zipfile

drive.mount("/content/drive")

MODEL_PATH     = "/content/drive/MyDrive/logseer/logseer.keras"
TOKENIZER_PATH = "/content/drive/MyDrive/logseer/tokenizer.pickle"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(f"Tokenizer file not found: {TOKENIZER_PATH}")

model = tf.keras.models.load_model(MODEL_PATH)

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("Loaded model and tokenizer successfully.")

In [ ]:
# Configuration
MAX_SEQUENCE_LENGTH = 26000
DRIVE_DATA_ZIP      = "/content/drive/MyDrive/Colab Notebooks/logseer/data/data.zip"
LOCAL_DATA_DIR      = "./predict_logs"

In [ ]:
# Load inference data from Google Drive zip
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

!rm -rf predict_logs predict_data.zip 2>/dev/null
shutil.copy(DRIVE_DATA_ZIP, "predict_data.zip")
with zipfile.ZipFile("predict_data.zip", "r") as z:
    z.extractall(LOCAL_DATA_DIR)

TEXT_DATA_DIR = os.path.join(LOCAL_DATA_DIR, "testdata")

texts, y_test, fnames = [], [], []
for dname in sorted(os.listdir(TEXT_DATA_DIR)):
    path = os.path.join(TEXT_DATA_DIR, dname)
    if os.path.isdir(path):
        label_id = 1 if dname == "error" else 0
        for fname in sorted(os.listdir(path)):
            with open(os.path.join(path, fname), encoding="utf-8") as f:
                texts.append(f.read())
            y_test.append(label_id)
            fnames.append(fname)

print(f"Loaded {len(texts)} files ({sum(y_test)} errors)")

text_seqs = tokenizer.texts_to_sequences(texts)
data = np.array(pad_sequences(text_seqs, maxlen=MAX_SEQUENCE_LENGTH), dtype=np.int32)

tester = Tester()
tester.testModel(model, data, y_test, fnames=fnames)